CORRECT

In [ ]:
# -----------------------------------------------
# Required Libraries
# -----------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.datasets import get_rdataset

# -----------------------------------------------
# Load Dataset
# -----------------------------------------------
# Load the Boston Housing dataset
boston = get_rdataset('Boston', 'MASS')
boston_df = boston.data

data_column = 'age'  # Age column for binning
num_bins = 3         # Number of bins for all binning operations

# -----------------------------------------------
# Type 1: Equal Frequency Binning (Manual)
# -----------------------------------------------
def equifreq(arr1, m):
    arr1 = sorted(arr1)
    a = len(arr1)
    n = int(a / m)
    for i in range(0, m):
        arr = arr1[i*n: (i+1)*n]
        print(f"Bin {i+1}: {arr}")

# -----------------------------------------------
# Type 2: Equal Width Binning (Manual)
# -----------------------------------------------
def equiwidth(arr1, m):
    w = (max(arr1) - min(arr1)) / m
    min_val = min(arr1)
    bins = [min_val + i * w for i in range(m + 1)]
    binned_data = [[] for _ in range(m)]

    for val in arr1:
        for i in range(m):
            if bins[i] <= val < bins[i+1]:
                binned_data[i].append(val)
                break
    for i, b in enumerate(binned_data):
        print(f"Bin {i+1}: {b}")

# Sample Data
data = [1, 3, 5, 7, 11, 15, 21, 30, 33, 51, 69, 81, 101, 111]

print("Equal Frequency Binning:")
equifreq(data, num_bins)

print("\nEqual Width Binning:")
equiwidth(data, num_bins)

# -----------------------------------------------
# Type 3: Equal Width Binning using np.digitize
# -----------------------------------------------
def equal_width_binning(data, num_bins):
    bin_edges = np.linspace(data.min(), data.max(), num_bins + 1)
    bins = np.digitize(data, bin_edges, right=False)
    return bins, bin_edges

boston_df['AGE_equal_width_bins'], _ = equal_width_binning(boston_df[data_column], num_bins)
display(boston_df[['age', 'AGE_equal_width_bins']].head(11))

# -----------------------------------------------
# Type 4: Equal Frequency Binning using pd.qcut
# -----------------------------------------------
def equal_freq_binning(data, num_bins):
    bins = pd.qcut(data, num_bins, labels=False, duplicates='drop')
    return bins

boston_df['AGE_equal_freq_bins'] = equal_freq_binning(boston_df[data_column], num_bins)
display(boston_df[['age', 'AGE_equal_freq_bins']].head(11))

# -----------------------------------------------
# Type 5: Smoothing by Bin Mean
# -----------------------------------------------
def bin_mean_smoothing(data, bins):
    bin_means = data.groupby(bins).transform('mean')
    return bin_means

boston_df['AGE_smoothed_mean'] = bin_mean_smoothing(boston_df[data_column], boston_df['AGE_equal_width_bins'])
display(boston_df[['age', 'AGE_smoothed_mean']].head(11))

# -----------------------------------------------
# Type 6: Smoothing by Bin Median
# -----------------------------------------------
def bin_median_smoothing(data, bins):
    bin_medians = data.groupby(bins).transform('median')
    return bin_medians

boston_df['AGE_smoothed_median'] = bin_median_smoothing(boston_df[data_column], boston_df['AGE_equal_width_bins'])
display(boston_df[['age', 'AGE_smoothed_median']].head(11))

# -----------------------------------------------
# Type 7: Smoothing by Bin Boundaries
# -----------------------------------------------
def bin_boundary_smoothing(data, bin_edges):
    bin_categories = pd.cut(data, bin_edges, right=False)
    smoothed_data = bin_categories.apply(lambda x: x.left).astype(float)
    return smoothed_data

bin_edges = np.linspace(boston_df[data_column].min(), boston_df[data_column].max(), num_bins + 1)
boston_df['AGE_smoothed_boundary'] = bin_boundary_smoothing(boston_df[data_column], bin_edges)
display(boston_df[['age', 'AGE_smoothed_boundary']].head(11))
